# Bias detection using the original data
---
The goal of this notebook is to use different tools to check whether automatic bias detection is possible. We apply it on the original data aswell to have comparable metrics

In [ ]:
import sys

sys.path.append("../src")  # go to parent dir
import warnings
import json
import pandas as pd
from sklearn.linear_model import LogisticRegression
from data_utils import split_data, apply_binning, apply_pdf, convert_intervals
from models_utils import (
    train_and_evaluate_pipeline,
)
from fairness_utils import (
    search_bias,
    evaluate_fairness_score,
    explain_bias,
    encode_protected_attributes,
    search_and_evaluate_fairness,
    train_and_evaluate_fairness_pipeline
)

warnings.filterwarnings("ignore")
random_state = 12041500

## Load data
We start by loading the respective data from the `./data` directory.

In [3]:
def load_data():
    df_train = pd.read_json("../data/trainset.json")
    df_test = pd.read_json("../data/testset.json")

    df_train = df_train.drop(columns=["fnlwgt"])
    df_test = df_test.drop(columns=["fnlwgt"])
    return df_train, df_test


ratio_features = ["age", "capital-gain", "capital-loss", "hours-per-week"]
ordinal_features = ["education-num"]
nominal_features = [
    "workclass",
    "marital-status",
    "occupation",
    "relationship",
    "race",
    "sex",
]
target = "income"

In [4]:
df_train, df_test = load_data()

## Baseline Classifier
This sections simply trains and evaluates our baseline. We simply use a Logistic Regression because it should be an easy but robust classifier.

In [5]:
clf = LogisticRegression(max_iter=1000, random_state=random_state)

In [6]:
## Train baseline
model, _ = train_and_evaluate_pipeline(
    clf, nominal_features, df_train, df_test, target, drop_na=True, verbose=True
)

Metric          : Value          
Accuracy        : 0.849
Precision       : 0.800
Recall          : 0.761
F1              : 0.777


## Fairness Evaluation
Next, we continue with the evaluation of fairness using multiple approaches.

In [7]:
X_train, y_train = split_data(df_train, target, drop_na=True)
probs = pd.Series(model.predict_proba(X_train)[:, 1])

### Search Bias
We use MDSS to perform a search for privileged classed, concerning the favorable label `>50k`.

In [8]:
privileged_subset, _ = search_bias(X_train, y_train, probs, 1, penalty=1)
print(privileged_subset)

({'capital-gain': [1409, 1424, 1455, 1506, 1797, 2105, 2174, 2202, 2228, 2290, 2329, 2346, 2354, 2407, 2414, 2463, 2580, 2597, 2635, 2653, 2829, 2885, 2907, 2936, 2961, 3137, 3273, 3325, 3411, 3432, 3456, 3464, 3471, 3674, 3781, 3818, 3908, 3942, 4064, 4101, 4416, 4508, 4650, 4865, 4931, 5013, 5455, 5721, 6360, 6497, 6723, 6767, 6849, 7443, 7978, 10566, 22040, 34095, 41310]}, 547.3998)


In [9]:
_ = evaluate_fairness_score(df_train, privileged_subset[0].keys(), target, verbose=True)

Sensitive Attributes: ['capital-gain']

Empty DataFrame
Columns: [Group, Distance, Proportion, Counts, P-Value]
Index: []

Weighted Mean Statistical Distance: nan


We inspect the subset size, expected probability, and classifier’s probability for protected attributes and privileged classes. We do this to get an better feel for the "problem size" we are facing.

In [10]:
explain_bias(df_train, probs, target, privileged_subset[0])

Our detected privileged group has a size of 984, we observe 0.0 as the average probability of earning >50k, but our model predicts 0.2604


## Fairness Metrics
We continue by calculating the fairness metrics: `SPD, EOD, AOD, DI` and `Theil index`. For this we use the AIF360 toolbox. We start by encoding the protected attributes as 1 or 0 based if the respective value is within a privileged class. Furthermore, we have to set those attributes as indices of the dataframe to make it work with the framework.

In [11]:
df_train_bias = encode_protected_attributes(df_train, list(privileged_subset[0].keys()), list(privileged_subset[0].values()), verbose=True)
df_test_bias = encode_protected_attributes(df_test, list(privileged_subset[0].keys()), list(privileged_subset[0].values()), verbose=True)

764 Na rows removed!
202 Na rows removed!


Lastly, we can compute the respective metrics

In [12]:
_, _ = train_and_evaluate_fairness_pipeline(
    clf,
    nominal_features,
    df_train_bias,
    target,
    privileged_subset,
    drop_na=True,
    verbose=True,
)

Metric                         : Value          
statistical_parity_difference   -0.125
average_odds_difference         0.175
equal_opportunity_difference    0.610
disparate_impact                0.615
theil_index                     0.120


Next, we continue by looking, how much the fairness metrics depend on the penaltiy parameter

In [121]:
df_fairness_metrics, priviliged_subsets = (
    pd.DataFrame(
        columns=[
            "statistical_parity_difference",
            "average_abs_odds_difference",
            "equal_opportunity_difference",
            "disparate_impact",
            "theil_index",
        ]
    ),
    {},
)

for penalty in [1e-17, 1e-10, 0.001, 0.01, 0.1, 0, 1, 5, 10, 25, 50, 100]:
    metrics, priv = search_and_evaluate_fairness(model, df_train, target, penalty)
    df_fairness_metrics.loc[f"{penalty}"] = metrics.values()
    priviliged_subsets[f"{penalty}"] = priv

In [122]:
df_fairness_metrics.to_json("../results/data/fairness_metrics_original.json")
with open("../results/data/fairness_privileged_classes_original.json", "w") as f:  
    json.dump(priviliged_subsets, f, default=convert_intervals, indent=4)

df_fairness_metrics

,statistical_parity_difference,average_abs_odds_difference,equal_opportunity_difference,disparate_impact,theil_index
1e-17,-0.066640,0.199234,0.600408,0.746289,0.121709
1e-10,-0.066640,0.199234,0.600408,0.746289,0.121709
0.001,-0.066640,0.199234,0.600408,0.746289,0.121709
0.01,-0.066640,0.199234,0.600408,0.746289,0.121709
0.1,-0.069909,0.197622,0.600408,0.737043,0.121709
0,-0.066640,0.199234,0.600408,0.746289,0.121709
1,-0.121794,0.171964,0.600408,0.615513,0.121709
5,-0.214702,0.126118,0.600408,0.475085,0.121709
10,-0.280192,0.093735,0.600408,0.409512,0.121709
25,-0.537335,-0.034031,0.600408,0.266332,0.121709


## Binned Data
Furthermore, we consider an approach given by AIF360 to use binning. For this, we take their approach 1:1 and check whether it has an influence.

In [123]:
nominal_features = nominal_features + ["age", "hours-per-week", "capital-gain", "capital-loss"]

In [124]:
df_train, df_test = load_data()
df_train = apply_binning(df_train.dropna())
df_test = apply_binning(df_test.dropna())

Next, we convert all columns with type category to object to avoid problems, which later approaches.

In [125]:
for col in df_train.columns:
    if df_train[col].dtype == "category":
        df_train[col] = df_train[col].astype("object")
        df_test[col] = df_test[col].astype("object")

## Baseline Classifier
This sections simply trains and evaluates our baseline. We simply use a Logistic Regression because it should be an easy but robust classifier.

In [126]:
clf = LogisticRegression(max_iter=1000, random_state=random_state)

In [128]:
## Train baseline
model, _ = train_and_evaluate_pipeline(
    clf, nominal_features, df_train, df_test, target, drop_na=True, verbose=True
)

Metric          : Value          
Accuracy        : 0.853
Precision       : 0.810
Recall          : 0.766
F1              : 0.784


## Fairness Evaluation
Next, we continue with the evaluation of fairness using multiple approaches.

In [129]:
X_train, y_train = split_data(df_train, target, drop_na=True)
probs = pd.Series(model.predict_proba(X_train)[:, 1])

### Search Bias
We use MDSS to perform a search for privileged classed, concerning the favorable label `>50k`.

In [130]:
privileged_subset, _ = search_bias(X_train, y_train, probs, 1, penalty=1)
print(privileged_subset)

({'hours-per-week': ['FullTime', 'MidTime', 'OverTime'], 'workclass': ['Federal-gov', 'Local-gov', 'Private'], 'relationship': ['Husband', 'Wife'], 'capital-loss': ['HighLoss']}, 19.2157)


In [131]:
_ = evaluate_fairness_score(df_train, privileged_subset[0].keys(), target, verbose=True)

Sensitive Attributes: ['capital-loss', 'hours-per-week', 'relationship', 'workclass']

                     Group Distance  Proportion  Counts   P-Value
        Own-child, Private    0.230    0.117701    4509  0.00e+00
                 Own-child    0.229    0.150852    5779  0.00e+00
 NoLoss, OverTime, Husband   -0.305    0.109948    4212  0.00e+00
         OverTime, Husband   -0.324    0.119189    4566  0.00e+00
          Husband, Private   -0.195    0.267352   10242 4.94e-324
           NoLoss, Husband   -0.190    0.381895   14630 4.94e-324
NoLoss, Own-child, Private    0.230    0.114986    4405 4.94e-324
                   Husband   -0.208    0.407685   15618 4.94e-324
         NoLoss, Own-child    0.230    0.147224    5640 4.94e-324
  NoLoss, Husband, Private   -0.177    0.250568    9599 1.20e-315

Weighted Mean Statistical Distance: 0.14188452864128123


We inspect the subset size, expected probability, and classifier’s probability for protected attributes and privileged classes. We do this to get an better feel for the "problem size" we are facing.

In [132]:
explain_bias(df_train, probs, target, privileged_subset[0])

Our detected privileged group has a size of 90, we observe 0.2222 as the average probability of earning >50k, but our model predicts 0.5559


## Fairness Metrics
We continue by calculating the fairness metrics: `SPD, EOD, AOD, DI` and `Theil index`. For this we use the AIF360 toolbox. We start by encoding the protected attributes as 1 or 0 based if the respective value is within a privileged class. Furthermore, we have to set those attributes as indices of the dataframe to make it work with the framework.

In [133]:
df_train_bias = encode_protected_attributes(df_train, list(privileged_subset[0].keys()), list(privileged_subset[0].values()), verbose=True)
df_test_bias = encode_protected_attributes(df_test, list(privileged_subset[0].keys()), list(privileged_subset[0].values()), verbose=True)

Lastly, we can compute the respective metrics

In [134]:
_, _ = train_and_evaluate_fairness_pipeline(
    clf,
    nominal_features,
    df_train_bias,
    target,
    privileged_subset,
    drop_na=True,
    verbose=True,
)

Metric                         : Value          
statistical_parity_difference   -0.347
average_odds_difference         -0.351
equal_opportunity_difference    -0.339
disparate_impact                0.363
theil_index                     0.118


Next, we continue by looking, how much the fairness metrics depend on the penaltiy parameter

In [135]:
df_fairness_metrics, priviliged_subsets = (
    pd.DataFrame(
        columns=[
            "statistical_parity_difference",
            "average_abs_odds_difference",
            "equal_opportunity_difference",
            "disparate_impact",
            "theil_index",
        ]
    ),
    {},
)

for penalty in [1e-17, 1e-10, 0.001, 0.01, 0.1, 0, 1, 5, 10, 25, 50, 100]:
    metrics, priv = search_and_evaluate_fairness(model, df_train, target, penalty)
    df_fairness_metrics.loc[f"{penalty}"] = metrics.values()
    priviliged_subsets[f"{penalty}"] = priv

In [136]:
df_fairness_metrics.to_json("../results/data/fairness_metrics_original_binning.json")
with open("../results/data/fairness_privileged_classes_original_binning.json", "w") as f:  
    json.dump(priviliged_subsets, f, default=convert_intervals, indent=4)
    
df_fairness_metrics

,statistical_parity_difference,average_abs_odds_difference,equal_opportunity_difference,disparate_impact,theil_index
1e-17,0.266412,0.358207,0.618180,655.640546,0.117754
1e-10,0.266384,0.358199,0.618180,655.371397,0.117754
0.001,0.266384,0.358199,0.618180,655.371397,0.117754
0.01,-0.285315,-0.384832,-0.388161,0.409693,0.117754
0.1,0.126427,0.175246,0.343735,2.724830,0.117754
0,0.266412,0.358207,0.618180,655.640546,0.117754
1,-0.346794,-0.351255,-0.338721,0.363031,0.117754
5,-0.424474,-0.311169,-0.279585,0.316125,0.117754
10,0.000000,0.000000,0.000000,1.000000,0.000000
25,0.000000,0.000000,0.000000,1.000000,0.000000


## PDFed Data
Additionally, we propose an alternative approach using PDFs to encode the same features. This method groups data based on the actual distribution rather than discrete bins, resulting in a smooth curve that represents the distribution continuously. This may offer
a potentially more effective solution.

In [137]:
nominal_features = nominal_features + ["age", "hours-per-week"]

In [138]:
df_train, df_test = load_data()
df_train = apply_pdf(df_train.dropna(), target)
df_test = apply_pdf(df_test.dropna(), target)

Next, we convert all columns with type category to object to avoid problems, which later approaches.

In [139]:
for col in df_train.columns:
    if df_train[col].dtype == "category":
        df_train[col] = df_train[col].astype("object")
        df_test[col] = df_test[col].astype("object")

### Baseline Classifier
This sections simply trains and evaluates our baseline. We simply use a Logistic Regression because it should be an easy but robust classifier.

In [140]:
clf = LogisticRegression(max_iter=1000, random_state=random_state)

In [142]:
## Train baseline
model, _ = train_and_evaluate_pipeline(
    clf, nominal_features, df_train, df_test, target, drop_na=True, verbose=True
)

Metric          : Value          
Accuracy        : 0.814
Precision       : 0.747
Recall          : 0.770
F1              : 0.757


### Fairness Evaluation
Next, we continue with the evaluation of fairness using multiple approaches.

In [143]:
X_train, y_train = split_data(df_train, target, drop_na=True)
probs = pd.Series(model.predict_proba(X_train)[:, 1])

### Search Bias
We use MDSS to perform a search for privileged classed, concerning the favorable label `>50k`.

In [144]:
privileged_subset, _ = search_bias(X_train, y_train, probs, 1, penalty=1)
print(privileged_subset)

({'capital-gain': [1.108469510907749e-06, 5.765883500686317e-06, 6.10361661605396e-06, 6.812113570428972e-06, 7.014401995094196e-06, 7.1738364190331214e-06, 7.194661828926502e-06, 7.660438928410234e-06, 7.947687124397542e-06, 8.667196401393174e-06, 8.825617371877361e-06, 8.867547263415679e-06, 8.974838428689815e-06, 9.017723059882917e-06, 9.383215882157025e-06, 9.392498484571787e-06, 9.466276197254861e-06, 9.603682302865236e-06, 1.0040343731611617e-05, 1.0775653765085864e-05, 1.0920757983701274e-05, 1.1258138748347011e-05, 1.170231032823149e-05, 1.3770732245764749e-05, 1.405480988072596e-05, 1.502609359416476e-05, 1.910843724102672e-05, 1.9325712488370405e-05, 2.1109643625328754e-05, 2.3619222072924518e-05, 2.6465815179851564e-05, 2.779276464904525e-05, 2.9315798274844017e-05, 3.351370696288232e-05, 0.00010547658630157628, 0.00012424286014113114]}, 34.1879)


In [145]:
_ = evaluate_fairness_score(df_train, privileged_subset[0].keys(), target, verbose=True)

Sensitive Attributes: ['capital-gain']

Empty DataFrame
Columns: [Group, Distance, Proportion, Counts, P-Value]
Index: []

Weighted Mean Statistical Distance: nan


We inspect the subset size, expected probability, and classifier’s probability for protected attributes and privileged classes. We do this to get an better feel for the "problem size" we are facing.

In [146]:
explain_bias(df_train, probs, target, privileged_subset[0])

Our detected privileged group has a size of 785, we observe 0.0 as the average probability of earning >50k, but our model predicts 0.2671


### Fairness Metrics
We continue by calculating the fairness metrics: `SPD, EOD, AOD, DI` and `Theil index`. For this we use the AIF360 toolbox. We start by encoding the protected attributes as 1 or 0 based if the respective value is within a privileged class. Furthermore, we have to set those attributes as indices of the dataframe to make it work with the framework.

In [147]:
df_train_bias = encode_protected_attributes(df_train, list(privileged_subset[0].keys()), list(privileged_subset[0].values()), verbose=True)
df_test_bias = encode_protected_attributes(df_test, list(privileged_subset[0].keys()), list(privileged_subset[0].values()), verbose=True)

Lastly, we can compute the respective metrics

In [148]:
_, _ = train_and_evaluate_fairness_pipeline(
    clf,
    nominal_features,
    df_train_bias,
    target,
    privileged_subset,
    drop_na=True,
    verbose=True,
)

Metric                         : Value          
statistical_parity_difference   0.188
average_odds_difference         0.343
equal_opportunity_difference    0.644
disparate_impact                15.792
theil_index                     0.106


Next, we continue by looking, how much the fairness metrics depend on the penaltiy parameter

In [149]:
df_fairness_metrics, priviliged_subsets = (
    pd.DataFrame(
        columns=[
            "statistical_parity_difference",
            "average_abs_odds_difference",
            "equal_opportunity_difference",
            "disparate_impact",
            "theil_index",
        ]
    ),
    {},
)

for penalty in [1e-17, 1e-10, 0.001, 0.01, 0.1, 0, 1, 5, 10, 25, 50, 100]:
    metrics, priv = search_and_evaluate_fairness(model, df_train, target, penalty)
    df_fairness_metrics.loc[f"{penalty}"] = metrics.values()
    priviliged_subsets[f"{penalty}"] = priv

In [150]:
df_fairness_metrics.to_json("../results/data/fairness_metrics_original_pdf.json")
with open("../results/data/fairness_privileged_classes_original_pdf.json", "w") as f:  
    json.dump(priviliged_subsets, f, default=convert_intervals, indent=4)

df_fairness_metrics

,statistical_parity_difference,average_abs_odds_difference,equal_opportunity_difference,disparate_impact,theil_index
1e-17,0.194652,0.345737,0.644437,23.677000,0.105923
1e-10,0.197982,0.348011,0.644437,55.709036,0.105923
0.001,0.197982,0.348011,0.644437,55.709036,0.105923
0.01,0.194652,0.345737,0.644437,23.677000,0.105923
0.1,0.194497,0.345680,0.644437,23.425549,0.105923
0,0.194652,0.345737,0.644437,23.677000,0.105923
1,0.188439,0.343284,0.644437,15.792466,0.105923
5,-0.085205,-0.076636,-0.085341,0.695386,0.105923
10,0.000000,0.000000,0.000000,1.000000,0.000000
25,0.000000,0.000000,0.000000,1.000000,0.000000
